# 02 Traces with pandas
Clean the traces, measure failure rates for two named segments, label a random and a stratified sample with a failure taxonomy, check PII redaction, and build a ranked failure backlog that includes your own Stage 1 probe failures.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-02"))
OUTPUT.mkdir(parents=True, exist_ok=True)
# Your Stage 1 probe runs (01_probe_set writes stage-01/probe_runs.csv). In Colab, upload it next to this notebook.
STAGE1 = Path(os.environ.get("KIT_STAGE1_DIR", ROOT / "stage-01"))
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: load the traces

In [2]:
import re
import pandas as pd
raw = pd.read_json(DATA / "traces.jsonl", lines=True)
print(len(raw), "rows")
# Raw message text holds PII, so it is never displayed: saved outputs stay safe to commit.
raw.drop(columns=["user_message", "assistant_output", "pii_planted"]).head()

615 rows


,trace_id,timestamp,user_id,segment,channel,outcome,latency_ms,cost_usd
0,t0001,2026-10-27 09:00:00+00:00,u176,free,web,resolved,6939,0.00243
1,t0002,2026-10-08 10:00:00+00:00,u656,free,web,resolved,615,0.00063
2,t0003,2026-10-01 11:00:00+00:00,u622,pro,android,escalated,8991,0.00082
3,t0004,2026-10-24 02:00:00+00:00,u552,pro,web,escalated,2129,0.00177
4,t0005,2026-10-17 18:00:00+00:00,u777,team,web,failed,1148,0.00059


Step 3: drop duplicates, empty outputs and test traffic

In [3]:
df = raw.drop_duplicates(subset="trace_id")
print("Duplicates removed:", len(raw) - len(df))
empty = df.assistant_output.str.strip() == ""
print("Empty outputs removed:", int(empty.sum()))
df = df[~empty]
test = df.user_id.str.startswith("test_")
print("Test traffic removed:", int(test.sum()))
df = df[~test].copy()
df["failed"] = df.outcome == "failed"
print(len(df), "rows left")

Duplicates removed: 15
Empty outputs removed: 12
Test traffic removed: 29
559 rows left


Step 4: failure rate for two named segments

A rate is only as honest as its numerator and denominator, so both are shown. Name your own two segments here.

In [4]:
SEGMENTS = {
    "team plan on android": (df.segment == "team") & (df.channel == "android"),
    "all other traffic": ~((df.segment == "team") & (df.channel == "android")),
}
rates = pd.DataFrame([{"segment": name, "failed traces (numerator)": int(df[mask].failed.sum()),
                       "all traces (denominator)": int(mask.sum()),
                       "failure rate": round(df[mask].failed.mean(), 3)} for name, mask in SEGMENTS.items()])
print("Every plan and channel, for context:")
print(df.groupby(["segment", "channel"]).failed.agg(failed="sum", traces="size", rate="mean").round(3))
rates

Every plan and channel, for context:
                 failed  traces   rate
segment channel                       
free    android       7      68  0.103
        ios          11      54  0.204
        web           9      59  0.153
pro     android       6      60  0.100
        ios           5      75  0.067
        web           6      62  0.097
team    android      38      66  0.576
        ios           2      50  0.040
        web           8      65  0.123


,segment,failed traces (numerator),all traces (denominator),failure rate
0,team plan on android,38,66,0.576
1,all other traffic,54,493,0.110


Step 5: the failure taxonomy

Each label has a meaning, a first guess at the layer (model, context, harness or surface), a default severity and a way to check a fix. Edit the labels to fit your product. `suggest_label` is only a starting rule: you review every label in Step 6.

In [5]:
TAXONOMY = {
    "ok": ("resolved with no problem seen", "", "", ""),
    "error_reply": ("the app replied with an error message instead of an answer", "harness", "high",
                    "replay these traces after the fix: no 'something went wrong' replies"),
    "escalated_billing": ("a billing or refund question that ended with a human", "context", "high",
                          "re-test 10 billing questions x 5 runs: no admin-only page cited, a clear support path"),
    "escalated_no_answer": ("a how-to question the assistant could not settle", "context", "medium",
                            "re-run these questions: the answer cites the right help page"),
    "slow_answer": ("resolved, but slower than 8 seconds", "harness", "low",
                    "re-run these questions: p90 latency under 8 seconds"),
    "probe_invalid_json": ("a Stage 1 probe asked for JSON and got text that does not parse", "harness", "medium",
                           "rerun the probe 5 times on the same model: 5 of 5 parse"),
    "probe_missing_word": ("a Stage 1 probe answer left out a required word", "model", "medium",
                           "rerun the probe 5 times on the same model: 5 of 5 contain the word"),
    "probe_format_broken": ("a Stage 1 probe answer broke the format rule", "model", "low",
                            "rerun the probe 5 times on the same model: 5 of 5 match the rule"),
}
pd.DataFrame(TAXONOMY, index=["meaning", "layer_guess", "severity", "how_to_check_fix"]).T

def suggest_label(row):
    if row.outcome == "failed":
        return "error_reply"
    if row.outcome == "escalated":
        return "escalated_billing" if "charged" in row.user_message.lower() else "escalated_no_answer"
    return "slow_answer" if row.latency_ms > 8000 else "ok"

df["failure_label"] = df.apply(suggest_label, axis=1)
df.failure_label.value_counts()

failure_label
ok                     278
escalated_no_answer    137
error_reply             92
slow_answer             35
escalated_billing       17
Name: count, dtype: int64

Step 6: label a random sample of 30 and a stratified sample of 30

The stratified sample takes 10 traces from each plan, so small groups are not missed. Read every row (Step 9 prints them redacted) and put any label you disagree with in `OVERRIDES` as `trace_id: label`, then run again.

In [6]:
OVERRIDES = {}  # for example {"t0042": "escalated_no_answer"}
assert set(OVERRIDES.values()) <= set(TAXONOMY), "Every override must be a label from TAXONOMY"
random_30 = df.sample(30, random_state=7).copy()
stratified_30 = df.groupby("segment").sample(n=10, random_state=7).copy()
for sample in (random_30, stratified_30):
    sample["failure_label"] = [OVERRIDES.get(t, label) for t, label in zip(sample.trace_id, sample.failure_label)]
print("Random sample by plan:", random_30.segment.value_counts().to_dict())
print("Stratified sample by plan:", stratified_30.segment.value_counts().to_dict())
pd.DataFrame({"random_30": random_30.failure_label.value_counts(),
              "stratified_30": stratified_30.failure_label.value_counts()}).fillna(0).astype(int)

Random sample by plan: {'pro': 14, 'team': 9, 'free': 7}
Stratified sample by plan: {'free': 10, 'pro': 10, 'team': 10}


,random_30,stratified_30
failure_label,,
ok,16,15
escalated_no_answer,8,6
error_reply,3,5
escalated_billing,2,2
slow_answer,1,2


Step 7: run a PII redaction pass on the sampled traces, and count caught and missed

The kit traces carry `pii_planted`, the true PII in each message, so you can review the redaction against it.

In [7]:
PATTERNS = {
    "card": r"\b(?:\d{4}[ -]?){3}\d{4}\b",
    "email": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone": r"\+?\d{0,2}\s?\d{3}-\d{3}-\d{4}",
}

def redact(text, patterns):
    for name, pattern in patterns.items():
        text = re.sub(pattern, f"[{name.upper()}]", text)
    return text

def pii_check(sample, patterns):
    checks = [(v, v not in redact(msg, patterns)) for msg, values in zip(sample.user_message, sample.pii_planted) for v in values]
    return sum(ok for _, ok in checks), [v for v, ok in checks if not ok]

sampled = pd.concat([random_30, stratified_30]).drop_duplicates("trace_id")
caught, missed = pii_check(sampled, PATTERNS)
print(f"First pass on {len(sampled)} sampled traces | planted: {caught + len(missed)} | caught: {caught} | missed: {len(missed)}")
print("Shapes it missed:", sorted({re.sub(r"\d", "9", v) for v in missed}))

First pass on 59 sampled traces | planted: 48 | caught: 34 | missed: 14
Shapes it missed: ['999.999.9999', 'user9999 at example dot com']


Step 8: fix the misses and check again

Add a pattern for each shape the first pass missed. Then the check runs on the sample and on every trace.

In [8]:
PATTERNS_V2 = {**PATTERNS,
    "phone_dots": r"\b\d{3}\.\d{3}\.\d{4}\b",
    "email_spelled": r"\b[\w.+-]+ at [\w-]+ dot \w+\b",
}
caught2, missed2 = pii_check(sampled, PATTERNS_V2)
all_caught, all_missed = pii_check(df, PATTERNS_V2)
print(f"Second pass on the samples | caught: {caught2} | missed: {len(missed2)}")
print(f"Second pass on all {len(df)} traces | caught: {all_caught} | missed: {len(all_missed)}")

def safe(text, planted):
    """Redacted text. If any true PII is still in it, withhold the whole text instead of saving it."""
    clean = redact(text, PATTERNS_V2)
    return "[WITHHELD: redaction missed PII]" if any(v in clean for v in planted) else clean

for frame in (df, random_30, stratified_30):
    frame["message_redacted"] = [safe(m, p) for m, p in zip(frame.user_message, frame.pii_planted)]
    frame["output_redacted"] = [safe(o, p) for o, p in zip(frame.assistant_output, frame.pii_planted)]
df[["trace_id", "message_redacted"]][df.pii_planted.str.len() > 0].head(3)

Second pass on the samples | caught: 48 | missed: 0
Second pass on all 559 traces | caught: 407 | missed: 0


,trace_id,message_redacted
2,t0003,Why was I charged twice? My details: [PHONE_DO...
3,t0004,The app crashes when I open a big note My deta...
4,t0005,"Why was I charged twice? My details: [CARD], [..."


Step 9: save the two labeled samples, redacted

Only redacted text is written to the stage folder, so these files are safe to commit.

In [9]:
COLS = ["trace_id", "segment", "channel", "outcome", "latency_ms", "message_redacted", "output_redacted", "failure_label"]
random_30[COLS].to_csv(OUTPUT / "labeled_random_30.csv", index=False)
stratified_30[COLS].to_csv(OUTPUT / "labeled_stratified_30.csv", index=False)
for name in ["labeled_random_30.csv", "labeled_stratified_30.csv"]:
    text = (OUTPUT / name).read_text()
    assert not any(v in text for v in df.pii_planted.explode().dropna()), f"PII found in {name}"
print("Saved", OUTPUT.name + "/labeled_random_30.csv and labeled_stratified_30.csv, no planted PII in either")
stratified_30[COLS].head(10)

Saved stage-02/labeled_random_30.csv and labeled_stratified_30.csv, no planted PII in either


,trace_id,segment,channel,outcome,latency_ms,message_redacted,output_redacted,failure_label
455,t0456,free,android,resolved,4280,"Where is dark mode? My details: [CARD], [CARD]",Here is how: Where is dark mode? My details...,ok
140,t0141,free,ios,resolved,2002,How do I undo a deleted note?,Here is how: How do I undo a deleted note?...,ok
564,t0565,free,android,escalated,3846,Why was I charged twice?,Here is how: Why was I charged twice?...,escalated_billing
480,t0481,free,web,failed,4886,How do I undo a deleted note?,"Sorry, something went wrong.",error_reply
419,t0420,free,android,escalated,4513,"Where is dark mode? My details: [PHONE_DOTS], ...",Here is how: Where is dark mode? My details...,escalated_no_answer
446,t0447,free,ios,resolved,3476,How do I export a note as PDF? My details: [PH...,Here is how: How do I export a note as PDF?...,ok
330,t0331,free,android,resolved,3216,Can I search inside images? My details: [EMAIL...,Here is how: Can I search inside images? My...,ok
108,t0109,free,web,escalated,756,How do I export a note as PDF? My details: [EM...,Here is how: How do I export a note as PDF?...,escalated_no_answer
402,t0403,free,android,escalated,2579,How do I undo a deleted note? My details: [EMA...,Here is how: How do I undo a deleted note? ...,escalated_no_answer
486,t0487,free,ios,escalated,2458,How do I export a note as PDF?,Here is how: How do I export a note as PDF?...,escalated_no_answer


Step 10: backlog rows from the kit traces

One row per failure label and plan and channel (3 or more traces). `trace_ids` lists up to 8 of the traces behind the row; `n_traces` is the full count.

In [10]:
cohort_rate = df.groupby(["segment", "channel"]).failed.mean().rename("cohort_failure_rate")
bad = df[df.failure_label != "ok"].join(cohort_rate, on=["segment", "channel"])
trace_rows = (bad.groupby(["failure_label", "segment", "channel"])
              .agg(trace_ids=("trace_id", lambda s: " ".join(s.head(8))), n_traces=("trace_id", "size"),
                   cohort_failure_rate=("cohort_failure_rate", "first"), example_message=("message_redacted", "first"))
              .reset_index())
trace_rows = trace_rows[trace_rows.n_traces >= 3]
trace_rows["from_stage1_probe"] = "no"
print(len(trace_rows), "backlog rows from the kit traces")

24 backlog rows from the kit traces


Step 11: add your own Stage 1 probe failures

This reads `probe_runs.csv` from your Stage 1 probe set notebook (in `stage-01/`, or uploaded next to this notebook in Colab). Each failed probe on each model becomes one row, marked `from_stage1_probe = yes`.

In [11]:
CHECK_LABEL = {"valid_json": "probe_invalid_json", "must_contain": "probe_missing_word", "regex": "probe_format_broken"}
found = [path for path in [STAGE1 / "probe_runs.csv", Path("probe_runs.csv")] if path.exists()]
probe_rows = pd.DataFrame()
if found:
    probe = pd.read_csv(found[0])
    fails = probe[~probe.passed.astype(bool)]
    probe_rows = (fails.groupby(["probe_id", "model", "check_type"])
                  .agg(trace_ids=("run", lambda s: " ".join(f"s1-{fails.loc[s.index[0], 'model']}-{fails.loc[s.index[0], 'probe_id']}-run{r}" for r in s)),
                       n_traces=("run", "size"), example_message=("text", "first"))
                  .reset_index())
    probe_rows["cohort_failure_rate"] = probe_rows.n_traces / probe.groupby(["probe_id", "model"]).size().max()
    probe_rows["failure_label"] = probe_rows.check_type.map(CHECK_LABEL)
    probe_rows["segment"], probe_rows["channel"] = "my Stage 1 probes", probe_rows.model
    probe_rows["from_stage1_probe"] = "yes"
    print("Read", found[0].name, "|", len(fails), "failed runs |", len(probe_rows), "backlog rows")
else:
    print("No Stage 1 probe_runs.csv found. Run notebooks/01_probe_set.ipynb, then put its probe_runs.csv in stage-01/",
          "(or upload it here in Colab) and run this cell again. The brief needs your Stage 1 failures in the backlog.")

Read probe_runs.csv | 21 failed runs | 8 backlog rows


Step 12: rank and save failure_backlog.csv

Rank: severity first, then how many traces, then the cohort failure rate. `layer_guess`, `severity` and `how_to_check_fix` start from the taxonomy: change any you disagree with in the saved CSV.

In [12]:
backlog = pd.concat([trace_rows, probe_rows], ignore_index=True)
backlog["layer_guess"] = backlog.failure_label.map(lambda l: TAXONOMY[l][1])
backlog["severity"] = backlog.failure_label.map(lambda l: TAXONOMY[l][2])
backlog["how_to_check_fix"] = backlog.failure_label.map(lambda l: TAXONOMY[l][3])
backlog["_sev"] = backlog.severity.map({"high": 0, "medium": 1, "low": 2})
backlog = backlog.sort_values(["_sev", "n_traces", "cohort_failure_rate"], ascending=[True, False, False])
backlog.insert(0, "rank", range(1, len(backlog) + 1))
backlog["cohort_failure_rate"] = backlog.cohort_failure_rate.round(3)
backlog = backlog[["rank", "trace_ids", "n_traces", "failure_label", "layer_guess", "severity", "how_to_check_fix",
                   "from_stage1_probe", "segment", "channel", "cohort_failure_rate", "example_message"]]
backlog.to_csv(OUTPUT / "failure_backlog.csv", index=False)
print("Saved", len(backlog), "rows to", OUTPUT.name + "/failure_backlog.csv |",
      (backlog.from_stage1_probe == "yes").sum(), "from your Stage 1 probes")
backlog.head(12)

Saved 32 rows to stage-02/failure_backlog.csv | 8 from your Stage 1 probes


,rank,trace_ids,n_traces,failure_label,layer_guess,severity,how_to_check_fix,from_stage1_probe,segment,channel,cohort_failure_rate,example_message
6,1,t0020 t0039 t0046 t0053 t0063 t0066 t0093 t0096,38,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,team,android,0.576,Can I share a notebook with my team?
1,2,t0156 t0161 t0184 t0233 t0250 t0361 t0425 t0518,11,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,free,ios,0.204,How do I undo a deleted note? My details: [EMA...
2,3,t0067 t0131 t0144 t0182 t0278 t0381 t0481 t0517,9,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,free,web,0.153,How do I undo a deleted note?
7,4,t0005 t0051 t0172 t0394 t0463 t0491 t0526 t0573,8,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,team,web,0.123,"Why was I charged twice? My details: [CARD], [..."
0,5,t0036 t0059 t0165 t0186 t0297 t0300 t0301,7,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,free,android,0.103,"My notes stopped syncing My details: [CARD], [..."
3,6,t0016 t0168 t0267 t0432 t0444 t0584,6,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,pro,android,0.100,The app crashes when I open a big note
5,7,t0044 t0098 t0159 t0253 t0496 t0557,6,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,pro,web,0.097,Can I share a notebook with my team?
4,8,t0115 t0191 t0260 t0345 t0437,5,error_reply,harness,high,replay these traces after the fix: no 'somethi...,no,pro,ios,0.067,How do I export a note as PDF? My details: [CARD]
8,9,t0264 t0565 t0577,3,escalated_billing,context,high,re-test 10 billing questions x 5 runs: no admi...,no,free,android,0.103,Why was I charged twice?
11,10,t0024 t0033 t0060 t0064 t0079 t0104 t0109 t0123,20,escalated_no_answer,context,medium,re-run these questions: the answer cites the r...,no,free,web,0.153,How do I undo a deleted note? My details: [PHO...


Step 13: README (copy this into `stage-02/README.md`, or keep it here if you submit notebook links)

## Decisions

## What I measured

## What I would do next

## Retention and access

| Where traces live | What is stored | How long I keep it | Who can access it | How it is deleted or cleared |
| --- | --- | --- | --- | --- |
| `data/traces.jsonl` in the kit | |  |  |  |
| this notebook's runtime (Colab or local) | |  |  |  |
| `stage-02/*.csv` in my GitHub repo | |  |  |  |
| `stage-01/probe_runs.csv` | |  |  |  |
| the model provider's logs | |  |  |  |